In [ ]:
import ee

# NOTA: "viirs-peru" es el ID del proyecto de Google Cloud vinculado a Earth Engine, el nombre fue ese pero es para todo EARTH ENGINE
# sirve como autenticación para CUALQUIER dataset de Earth Engine (WorldCover, SRTM,
# Sentinel-2, Open Buildings, etc.), no solo para VIIRS.

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")

# Datos administrativos y municipales

## RENAMU (Registro Nacional de Municipalidades)

**Fuente:** https://proyectos.inei.gob.pe/microdatos/ (buscar "RENAMU" — desde 2021 la encuesta se estandariza en un único módulo llamado "Registro Nacional de Municipalidades - RENAMU", sin división por módulos como en años anteriores)

RENAMU es la encuesta anual que aplica el INEI a todas las municipalidades del país. Acá se usa como fuente de variables de control para el modelo causal (Etapa 6, DML + Causal Forest): el riesgo que cubre es que el modelo confunda "el gasto no rinde porque el territorio no responde" con "el gasto no rinde porque la municipalidad gestiona mal". Sin este control, τ podría estar capturando capacidad de gestión municipal en vez de contexto territorial — que es justo lo que el proyecto busca aislar.

### Por qué el panel arranca en 2021 y no en 2020

- El módulo con la pregunta sobre anemia (P68_7) recién aparece en la encuesta 2021. RENAMU 2020 no la tiene, así que no sirve como punto de partida.
- Desde 2021 el formato está estandarizado en un único módulo: mismos nombres de columna año a año (confirmado revisando el portal directamente, carpeta por carpeta).
- Desde 2021 el propio CSV trae la columna `Ubigeo` ya limpia en la cabecera. En 2020 solo existe `idmunici`, y hay que reconstruir el ubigeo a mano con `.zfill(6)` — trabajo extra que no vale la pena si igual ese año no trae la variable que más importa.

Por eso el panel se arma como loop 2021 → último año disponible, con el mismo código para cada año, en vez de tratar 2020 como caso especial.

### Variables extraídas (panel por Ubigeo + Año)

| Variable | Código original | Nombre final | Redacción exacta del diccionario (RENAMU 2021) | Corrección de año |
|---|---|---|---|---|
| Personal municipal | P19D_T | `personal_total` | *"Total personal / 31 de diciembre 2020"* — dentro del bloque "Personal de la municipalidad, al 31 de diciembre 2020". Es un conteo, no Sí/No. | **Sí, retrospectiva.** El campo describe el personal al cierre del año anterior a la encuesta, no del año de la encuesta. Etiquetar como año = X−1 |
| Programa de prevención de anemia con MINSA | P68_7 | `programa_anemia` | *"En el año 2020, ¿La municipalidad implementó programas de control y prevención de la salud en coordinación con el MINSA en: Prevención y reducción de la anemia"* — ítem dentro de un checklist de 11 opciones (P68_1 a P68_11), no un Sí/No binario simple. Verificar valores únicos reales en el CSV antes de tratarla como booleana. | Sí, retrospectiva: encuesta año X pregunta por lo ejecutado en X−1. Etiquetar como año = X−1 |
| Centro de salud administrado por la municipalidad | P66_2 | `centro_salud_municipal` | *"¿En el Distrito funcionan establecimientos de salud administrados por la municipalidad: Centro de salud?"* — 1: Sí / 2: No | No. Pregunta en tiempo presente, sin referencia retrospectiva → el valor corresponde al mismo año de la encuesta |

### Corrección de año: las tres variables no llevan el mismo tratamiento

A diferencia de lo que se asumió inicialmente, **dos de las tres variables son retrospectivas, no solo una**:

- `personal_total` (P19D_T) y `programa_anemia` (P68_7) describen la situación del año **anterior** a la encuesta → se etiquetan con año = Año_encuesta − 1.
- `centro_salud_municipal` (P66_2) describe la situación **al momento de la encuesta** → se etiqueta con año = Año_encuesta, sin ajuste.

Si esta corrección no se aplica correctamente a cada variable, el merge con anemia (SIEN) y gasto (SIAF) queda desfasado, y el modelo terminaría comparando información municipal de un año con el gasto/anemia de otro año distinto.

### Pendiente antes de dar por cerrada la tabla

- Confirmar con `df["programa_anemia"].value_counts()` qué valores únicos trae realmente esa columna en el CSV cargado — el diccionario sugiere que es un ítem de checklist (0/Pase, 9/Sí), no un Sí/No de dos valores limpio como se pensaba.

In [ ]:
from pathlib import Path
import pandas as pd

base_dir = Path("../data/raw/RENAMU")
años = range(2021, 2025)

columnas_necesarias = {
    "Ubigeo": "ubigeo",
    "P19D_T": "personal_total",
    "P68_7": "programa_anemia",
    "P66_2": "centro_salud_municipal",
}

nulos_renamu = ["#Â¡NULO!", "#¡NULO!"]

paneles = []

for año in años:
    carpeta = base_dir / str(año)

    if not carpeta.exists():
        print(f"⚠️ {año}: no existe la carpeta {carpeta}, saltando")
        continue

    csvs = [f for f in carpeta.iterdir() if f.suffix.lower() == ".csv"]

    if len(csvs) != 1:
        print(f"⚠️ {año}: encontré {len(csvs)} CSV en la carpeta, revisar manualmente")
        continue

    df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)

    faltantes = [c for c in columnas_necesarias if c not in df.columns]
    if faltantes:
        print(f"⚠️ {año}: faltan columnas {faltantes} — revisar nombre exacto en el diccionario {año}")
        print(f"   Columnas disponibles (primeras 15): {list(df.columns)[:15]}")
        continue

    df = df[list(columnas_necesarias.keys())].rename(columns=columnas_necesarias)
    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    df["Año_encuesta"] = año

    # Diagnóstico rápido: confirmar qué valores trae programa_anemia antes de asumir Sí/No
    print(f"{año} — valores únicos en programa_anemia: {df['programa_anemia'].unique()}")

    paneles.append(df)
    print(f"✅ {año}: {df.shape[0]} municipalidades cargadas")

renamu_panel = pd.concat(paneles, ignore_index=True)

# --- Corrección de año: personal_total y programa_anemia son retrospectivas ---
# (describen el año anterior a la encuesta); centro_salud_municipal no lo es.
renamu_panel["Año_personal_total"] = renamu_panel["Año_encuesta"] - 1
renamu_panel["Año_programa_anemia"] = renamu_panel["Año_encuesta"] - 1
# centro_salud_municipal usa Año_encuesta directamente, sin columna adicional

print(renamu_panel.shape)
renamu_panel.head()

C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_4_4_O, 1: P37A_5_O, 2: P73_4_O, 3: P95_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2021 — valores únicos en programa_anemia: [0 7]
✅ 2021: 1874 municipalidades cargadas


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_1_4_O, 1: P23_4_4_O, 2: P23_9_4_O, 3: P23_11_4_O, 4: P31_5_O, 5: P37A_5_O, 6: P73_4_O, 7: P79A_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2022 — valores únicos en programa_anemia: [0 7]
✅ 2022: 1874 municipalidades cargadas


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_14_1_O, 1: P23_14_2, 2: P28_5_O, 3: P37A_5_O, 4: P48_12_O, 5: P55_7_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2023 — valores únicos en programa_anemia: [ 7.  0. nan]
✅ 2023: 1891 municipalidades cargadas
2024 — valores únicos en programa_anemia: [0 7]
✅ 2024: 1891 municipalidades cargadas
(7530, 7)


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_13_2, 1: P25_4_O, 2: P28_5_O, 3: P48_12_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta,Año_personal_total,Año_programa_anemia
0,010101,208.0,0.0,2.0,2021,2020,2020
1,010102,1.0,0.0,2.0,2021,2020,2020
2,010103,7.0,0.0,2.0,2021,2020,2020
3,010104,5.0,7.0,2.0,2021,2020,2020
4,010105,2.0,0.0,2.0,2021,2020,2020


In [ ]:
from pathlib import Path
import pandas as pd

base_dir = Path("../data/raw/RENAMU")
años = range(2021, 2025)  # ajustar según lo que confirmes disponible en el portal

columnas_necesarias = {
    "Ubigeo": "ubigeo",
    "P19D_T": "personal_total",
    "P68_7": "programa_anemia",
    "P66_2": "centro_salud_municipal",
}

nulos_renamu = ["#Â¡NULO!", "#¡NULO!"]

paneles = []

for año in años:
    carpeta = base_dir / str(año)
    if not carpeta.exists():
        continue

    csvs = [f for f in carpeta.iterdir() if f.suffix.lower() == ".csv"]
    if len(csvs) != 1:
        continue

    df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)

    if any(c not in df.columns for c in columnas_necesarias):
        continue

    df = df[list(columnas_necesarias.keys())].rename(columns=columnas_necesarias)
    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    df["Año_encuesta"] = año

    paneles.append(df)

renamu_panel = pd.concat(paneles, ignore_index=True)

# --- Tres tablas filtradas, cada una con SOLO su variable y SU año real ---
# (listas para mergear después, cuando tengas tabla_maestra armada)

centro_salud = renamu_panel[["ubigeo", "Año_encuesta", "centro_salud_municipal"]].rename(
    columns={"Año_encuesta": "Año"}
)

personal = renamu_panel[["ubigeo", "Año_encuesta", "personal_total"]].copy()
personal["Año"] = personal["Año_encuesta"] - 1
personal = personal[["ubigeo", "Año", "personal_total"]]

anemia_muni = renamu_panel[["ubigeo", "Año_encuesta", "programa_anemia"]].copy()
anemia_muni["Año"] = anemia_muni["Año_encuesta"] - 1
anemia_muni = anemia_muni[["ubigeo", "Año", "programa_anemia"]]

print(renamu_panel.shape)
renamu_panel.head()

C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_4_4_O, 1: P37A_5_O, 2: P73_4_O, 3: P95_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)
C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_1_4_O, 1: P23_4_4_O, 2: P23_9_4_O, 3: P23_11_4_O, 4: P31_5_O, 5: P37A_5_O, 6: P73_4_O, 7: P79A_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)
C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_14_1_O, 1: P23_14_2, 2: P28_5_O, 3: P37A_5_O, 4: P48_12_O, 5: P55_7_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


(7530, 5)


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_13_2, 1: P25_4_O, 2: P28_5_O, 3: P48_12_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta
0,010101,208.0,0.0,2.0,2021
1,010102,1.0,0.0,2.0,2021
2,010103,7.0,0.0,2.0,2021
3,010104,5.0,7.0,2.0,2021
4,010105,2.0,0.0,2.0,2021


### Notas de codificación y tipos de dato

**`centro_salud_municipal` (P66_2):** es un indicador binario (Sí/No), no una cantidad. Corresponde específicamente al tipo de establecimiento "Centro de salud" administrado por la municipalidad — no incluye hospitales, postas, consultorios ni otros tipos (esos son preguntas separadas en el diccionario: P66_1, P66_3, P66_4, etc.). El campo de conteo real (`P66_2_1`, "Número de establecimientos") no forma parte de este panel.

**`programa_anemia` (P68_7):** el diccionario de RENAMU codifica esta pregunta como parte de un checklist de 11 programas de salud (P68_1 a P68_11), donde cada ítem usa su propia posición como código de "Sí" en vez de un 1/2 estándar. Para P68_7 específicamente: `0 = Pase` (no marcó esta opción) y `7 = Sí` (sí implementó el programa de anemia). Antes de usar esta variable en el modelo causal, se recodifica a booleano estándar (1 = Sí, 0 = No) para que no se interprete como una magnitud numérica.

**Tipos de dato:** las tres variables llegan como `float64` por los valores nulos (`NaN`) presentes en el CSV crudo. Se mantienen como `float64` hasta el merge final — convertir a `int` antes de imputar/tratar los nulos generaría un error, porque `NaN` no es representable como entero en pandas.

In [ ]:
# --- Recodificación a booleano estándar (1 = Sí, 0 = No) ---

# programa_anemia: el diccionario de RENAMU codifica "Sí" como 7 (la posición
# del ítem "anemia" dentro del checklist P68_1 a P68_11), no como 1.
# Se recodifica para que el modelo no lo interprete como una magnitud.
renamu_panel["programa_anemia"] = (renamu_panel["programa_anemia"] == 7).astype("Int64")

# centro_salud_municipal: viene como 1=Sí, 2=No (estándar RENAMU).
# Se recodifica a 1=Sí, 0=No para mantener consistencia con programa_anemia.
renamu_panel["centro_salud_municipal"] = (renamu_panel["centro_salud_municipal"] == 1).astype("Int64")

# personal_total: es un conteo real (no booleano). Se pasa a entero nullable
# porque tiene NaN, y un int64 normal de numpy no admite nulos.
renamu_panel["personal_total"] = renamu_panel["personal_total"].astype("Int64")

# --- Reconstruir las tres tablas filtradas con los valores ya recodificados ---

centro_salud = renamu_panel[["ubigeo", "Año_encuesta", "centro_salud_municipal"]].rename(
    columns={"Año_encuesta": "Año"}
)

personal = renamu_panel[["ubigeo", "Año_encuesta", "personal_total"]].copy()
personal["Año"] = personal["Año_encuesta"] - 1
personal = personal[["ubigeo", "Año", "personal_total"]]

anemia_muni = renamu_panel[["ubigeo", "Año_encuesta", "programa_anemia"]].copy()
anemia_muni["Año"] = anemia_muni["Año_encuesta"] - 1
anemia_muni = anemia_muni[["ubigeo", "Año", "programa_anemia"]]

print(renamu_panel.dtypes)
renamu_panel.head()

ubigeo                      str
personal_total            Int64
programa_anemia           Int64
centro_salud_municipal    Int64
Año_encuesta              int64
dtype: object


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta
0,010101,208,0,0,2021
1,010102,1,0,0,2021
2,010103,7,0,0,2021
3,010104,5,1,0,2021
4,010105,2,0,0,2021
